# Autoencoder Fallback Experiment

## Goal
Instead of using a global Bayesian average whenever album-level personal signal is missing,
we train a **track autoencoder** on direct track ratings from `train_data.csv` and use its
predicted track preference as the fallback signal.

## Why this is worth testing
In the heuristic notebook, a large share of candidate rows eventually fall back to a global
community prior. That is stable, but not very personal. An autoencoder can learn latent user
and track structure from listening/rating patterns and provide a **user-specific** estimate.

## What this notebook does
1. Build a local validation split from `train_data.csv`
2. Train an autoencoder on direct track ratings from the training split only
3. Compare two pipelines on the held-out validation tracks:
   - `Global Fallback`: direct -> album sibling -> global Bayesian
   - `Autoencoder Fallback`: direct -> album sibling -> autoencoder track score -> global Bayesian
4. Report ranking accuracy, exact-match rate, tier usage, and fallback diagnostics

## Important note
This autoencoder is being used as a **fallback / rescue signal**, not as a full replacement for
your whole hierarchy.


In [ ]:
import math
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.sparse import csr_matrix
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MIN_TRACK_RATINGS = 12
TOP_POPULAR_TRACKS = 12000
INCLUDE_TEST_CANDIDATES_IN_AE = False
AE_HIDDEN_DIM = 256
AE_LATENT_DIM = 64
AE_DROPOUT = 0.15
AE_EPOCHS = 10
AE_BATCH_SIZE = 256
AE_LR = 1e-3
AE_WEIGHT_DECAY = 1e-5
AE_BLEND_WEIGHT = 0.75

print('Setup complete')
print(f'  Device                     : {DEVICE}')
print(f'  Random seed                : {RANDOM_SEED}')
print(f'  Minimum track ratings/user : {MIN_TRACK_RATINGS}')
print(f'  Top popular tracks in AE   : {TOP_POPULAR_TRACKS:,}')
print(f'  Include test candidates    : {INCLUDE_TEST_CANDIDATES_IN_AE}')
print(f'  AE architecture            : {AE_HIDDEN_DIM} -> {AE_LATENT_DIM} -> {AE_HIDDEN_DIM}')


Setup complete
  Device                     : cpu
  Random seed                : 42
  Minimum track ratings/user : 12
  Top popular tracks in AE   : 12,000
  Include test candidates    : False
  AE architecture            : 256 -> 64 -> 256


## Load Data

We load the same core assets used in the heuristic notebook.


In [4]:
track_df = pd.read_csv('Assets/CSV/track_data.csv')
train_df = pd.read_csv('Assets/CSV/train_data.csv')
test_df = pd.read_csv('Assets/CSV/test_data.csv')

print(f'Tracks : {len(track_df):>10,}')
print(f'Train  : {len(train_df):>10,}')
print(f'Test   : {len(test_df):>10,}')

all_track_ids = set(track_df['TrackID'].dropna().astype(int).astype(str))
train_df['ItemID_str'] = train_df['ItemID'].astype(int).astype(str)
track_ratings_mask = train_df['ItemID_str'].isin(all_track_ids)

print()
print(f'Direct track-rating rows   : {track_ratings_mask.sum():>10,}  ({track_ratings_mask.mean()*100:.1f}%)')
print(f'Non-track item rows        : {(~track_ratings_mask).sum():>10,}  ({(~track_ratings_mask).mean()*100:.1f}%)')
print(f'Unique test candidate tracks: {test_df["TrackID"].nunique():,}')


Tracks :    224,041
Train  : 12,403,575
Test   :    120,000

Direct track-rating rows   :  5,480,041  (44.2%)
Non-track item rows        :  6,923,534  (55.8%)
Unique test candidate tracks: 48,832


## Static Hierarchy Maps

These are built from `track_data.csv` only and are safe to reuse across splits.


In [5]:
genre_cols = [c for c in track_df.columns if c.startswith('Genre')]

track_to_hierarchy = {}
album_to_tracks = defaultdict(list)
artist_to_tracks = defaultdict(list)

for _, row in tqdm(track_df.iterrows(), total=len(track_df), desc='Building hierarchy maps'):
    track_id = str(int(row['TrackID']))
    album_id = str(int(row['AlbumID'])) if pd.notna(row['AlbumID']) else None
    artist_id = str(int(row['ArtistID'])) if pd.notna(row['ArtistID']) else None
    genres = [str(int(row[c])) for c in genre_cols if pd.notna(row[c])]

    track_to_hierarchy[track_id] = {
        'album': album_id,
        'artist': artist_id,
        'genres': genres,
    }

    if album_id:
        album_to_tracks[album_id].append(track_id)
    if artist_id:
        artist_to_tracks[artist_id].append(track_id)

print(f'Hierarchy built for {len(track_to_hierarchy):,} tracks')
print(f'Albums with tracks  : {len(album_to_tracks):,}')
print(f'Artists with tracks : {len(artist_to_tracks):,}')


Building hierarchy maps:   0%|          | 0/224041 [00:00<?, ?it/s]

Hierarchy built for 224,041 tracks
Albums with tracks  : 31,141
Artists with tracks : 11,851


## Local Validation Split

We mimic the Kaggle task using only `train_data.csv`.

For each eligible user:
- hold out 6 directly rated tracks
- top 3 by rating become label `1`
- bottom 3 by rating become label `0`
- remove those 6 track rows from the training split

That gives us real labels for offline comparison.


In [6]:
track_ratings_df = train_df[track_ratings_mask].copy()
user_track_counts = track_ratings_df.groupby('UserID').size()
eligible_users = user_track_counts[user_track_counts >= MIN_TRACK_RATINGS].index

val_records = []
holdout_index = []

for uid, group in tqdm(
    track_ratings_df[track_ratings_df['UserID'].isin(eligible_users)].groupby('UserID'),
    total=len(eligible_users),
    desc='Creating validation split'
):
    sorted_group = group.sort_values('Rating', ascending=False)
    top3 = sorted_group.head(3)
    bottom3 = sorted_group.tail(3)

    holdout_index.extend(top3.index.tolist())
    holdout_index.extend(bottom3.index.tolist())

    for _, row in top3.iterrows():
        val_records.append({
            'UserID': int(uid),
            'TrackID': str(int(row['ItemID'])),
            'true_rating': float(row['Rating']),
            'true_label': 1,
        })
    for _, row in bottom3.iterrows():
        val_records.append({
            'UserID': int(uid),
            'TrackID': str(int(row['ItemID'])),
            'true_rating': float(row['Rating']),
            'true_label': 0,
        })

val_df = pd.DataFrame(val_records)
train_split = train_df.drop(index=holdout_index).copy()
train_split['ItemID_str'] = train_split['ItemID'].astype(int).astype(str)

print(f'Eligible users          : {len(eligible_users):,}')
print(f'Validation rows         : {len(val_df):,}')
print(f'Validation users        : {val_df["UserID"].nunique():,}')
print(f'Train split rows        : {len(train_split):,}')
print() 
print('Validation label counts:')
print(val_df['true_label'].value_counts().sort_index().to_string())

liked = val_df[val_df['true_label'] == 1]['true_rating']
disliked = val_df[val_df['true_label'] == 0]['true_rating']
print()
print('Held-out rating summary:')
print(f'  liked mean    : {liked.mean():.2f}')
print(f'  disliked mean : {disliked.mean():.2f}')
print(f'  gap           : {(liked.mean() - disliked.mean()):.2f}')


Creating validation split:   0%|          | 0/21800 [00:00<?, ?it/s]

Eligible users          : 21,800
Validation rows         : 130,800
Validation users        : 21,800
Train split rows        : 12,272,775

Validation label counts:
true_label
0    65400
1    65400

Held-out rating summary:
  liked mean    : 90.49
  disliked mean : 9.73
  gap           : 80.76


## Training Lookups from `train_split` Only


In [26]:
user_to_ratings = defaultdict(dict)
for _, row in tqdm(train_split.iterrows(), total=len(train_split), desc='Building user rating lookup'):
    user_to_ratings[str(int(row['UserID']))][str(int(row['ItemID']))] = float(row['Rating'])

overall_mean = float(train_split['Rating'].mean())

item_stats = (
    train_split.groupby('ItemID')['Rating']
    .agg(['mean', 'count'])
    .reset_index()
)
item_stats.columns = ['ItemID', 'global_mean', 'rating_count']
item_stats['ItemID'] = item_stats['ItemID'].astype(int).astype(str)

C = float(item_stats['rating_count'].median())
m = overall_mean
item_stats['bayesian_avg'] = (
    (item_stats['rating_count'] * item_stats['global_mean'] + C * m) /
    (item_stats['rating_count'] + C)
)

global_item_scores = dict(zip(item_stats['ItemID'], item_stats['bayesian_avg']))

track_stats = (
    train_split[train_split['ItemID_str'].isin(all_track_ids)]
    .groupby('ItemID')['Rating']
    .agg(['mean', 'count'])
    .reset_index()
)
track_stats.columns = ['TrackID', 'global_mean', 'rating_count']
track_stats['TrackID'] = track_stats['TrackID'].astype(int).astype(str)
track_stats['bayesian_avg'] = (
    (track_stats['rating_count'] * track_stats['global_mean'] + C * m) /
    (track_stats['rating_count'] + C)
)
global_track_scores = dict(zip(track_stats['TrackID'], track_stats['bayesian_avg']))

print(f'User lookup built for       : {len(user_to_ratings):,} users')
print(f'Overall mean rating         : {overall_mean:.2f}')
print(f'Global Bayesian item scores : {len(global_item_scores):,}')
print(f'Global track scores         : {len(global_track_scores):,}')
print(f'Bayesian confidence C       : {C:.1f}')


Building user rating lookup:   0%|          | 0/12272775 [00:00<?, ?it/s]

User lookup built for       : 49,204 users
Overall mean rating         : 49.77
Global Bayesian item scores : 295,746
Global track scores         : 223,727
Bayesian confidence C       : 10.0


## Autoencoder Training Data

We train a **track-level** autoencoder on direct track ratings only.

To keep the matrix manageable while still covering candidate tracks, the track vocabulary is:
- all validation tracks
- optionally all test candidate tracks
- top popular direct tracks from the training split


In [8]:
train_track_rows = train_split[train_split['ItemID_str'].isin(all_track_ids)].copy()
val_track_ids = set(val_df['TrackID'].astype(int).tolist())
popular_track_ids = set(
    train_track_rows['ItemID'].value_counts().head(TOP_POPULAR_TRACKS).index.astype(int).tolist()
)
ae_track_ids = set(val_track_ids) | set(popular_track_ids)

if INCLUDE_TEST_CANDIDATES_IN_AE:
    ae_track_ids |= set(test_df['TrackID'].astype(int).unique().tolist())

ae_track_ids = sorted(ae_track_ids)
ae_track_id_str = set(str(t) for t in ae_track_ids)

train_track_rows = train_track_rows[train_track_rows['ItemID_str'].isin(ae_track_id_str)].copy()
ae_user_ids = sorted(train_track_rows['UserID'].astype(int).unique().tolist())
user_to_idx = {uid: i for i, uid in enumerate(ae_user_ids)}
track_to_idx = {tid: j for j, tid in enumerate(ae_track_ids)}
idx_to_track = {j: tid for tid, j in track_to_idx.items()}

row_idx = train_track_rows['UserID'].astype(int).map(user_to_idx).to_numpy()
col_idx = train_track_rows['ItemID'].astype(int).map(track_to_idx).to_numpy()
values = (train_track_rows['Rating'].astype(np.float32) / 100.0).to_numpy()

ratings_csr = csr_matrix((values, (row_idx, col_idx)), shape=(len(ae_user_ids), len(ae_track_ids)), dtype=np.float32)
mask_csr = ratings_csr.copy()
mask_csr.data = np.ones_like(mask_csr.data, dtype=np.float32)

all_row_ids = np.arange(ratings_csr.shape[0])
np.random.shuffle(all_row_ids)
val_cut = max(1, int(0.10 * len(all_row_ids)))
ae_valid_rows = np.sort(all_row_ids[:val_cut])
ae_train_rows = np.sort(all_row_ids[val_cut:])

print(f'AE users                  : {len(ae_user_ids):,}')
print(f'AE track vocabulary       : {len(ae_track_ids):,}')
print(f'Observed ratings in AE    : {ratings_csr.nnz:,}')
print(f'Density                   : {ratings_csr.nnz / (ratings_csr.shape[0] * ratings_csr.shape[1]):.6f}')
print(f'AE train rows             : {len(ae_train_rows):,}')
print(f'AE valid rows             : {len(ae_valid_rows):,}')


AE users                  : 34,376
AE track vocabulary       : 50,213
Observed ratings in AE    : 3,399,955
Density                   : 0.001970
AE train rows             : 30,939
AE valid rows             : 3,437


In [9]:
def iter_dense_batches(ratings_matrix, mask_matrix, row_ids, batch_size, shuffle=True):
    row_ids = np.array(row_ids, copy=True)
    if shuffle:
        np.random.shuffle(row_ids)
    for start in range(0, len(row_ids), batch_size):
        batch_ids = row_ids[start:start + batch_size]
        x = torch.tensor(ratings_matrix[batch_ids].toarray(), dtype=torch.float32, device=DEVICE)
        mask = torch.tensor(mask_matrix[batch_ids].toarray(), dtype=torch.float32, device=DEVICE)
        yield batch_ids, x, mask


class TrackAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, latent_dim=64, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


model = TrackAutoencoder(
    input_dim=len(ae_track_ids),
    hidden_dim=AE_HIDDEN_DIM,
    latent_dim=AE_LATENT_DIM,
    dropout=AE_DROPOUT,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=AE_LR, weight_decay=AE_WEIGHT_DECAY)


def masked_mse(pred, target, mask):
    diff = (pred - target) * mask
    denom = torch.clamp(mask.sum(), min=1.0)
    return (diff.pow(2).sum()) / denom


history = []
for epoch in range(1, AE_EPOCHS + 1):
    model.train()
    train_losses = []
    for _, batch_x, batch_mask in iter_dense_batches(ratings_csr, mask_csr, ae_train_rows, AE_BATCH_SIZE, shuffle=True):
        optimizer.zero_grad()
        pred = model(batch_x)
        loss = masked_mse(pred, batch_x, batch_mask)
        loss.backward()
        optimizer.step()
        train_losses.append(float(loss.item()))

    model.eval()
    valid_losses = []
    with torch.no_grad():
        for _, batch_x, batch_mask in iter_dense_batches(ratings_csr, mask_csr, ae_valid_rows, AE_BATCH_SIZE, shuffle=False):
            pred = model(batch_x)
            loss = masked_mse(pred, batch_x, batch_mask)
            valid_losses.append(float(loss.item()))

    train_loss = float(np.mean(train_losses)) if train_losses else float('nan')
    valid_loss = float(np.mean(valid_losses)) if valid_losses else float('nan')
    history.append({'epoch': epoch, 'train_loss': train_loss, 'valid_loss': valid_loss})
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.6f} | valid_loss={valid_loss:.6f}')

history_df = pd.DataFrame(history)
print() 
print('Training history:')
display(history_df)


Epoch 01 | train_loss=0.126324 | valid_loss=0.118929
Epoch 02 | train_loss=0.111126 | valid_loss=0.110905
Epoch 03 | train_loss=0.095951 | valid_loss=0.107880
Epoch 04 | train_loss=0.086059 | valid_loss=0.106011
Epoch 05 | train_loss=0.080185 | valid_loss=0.103754
Epoch 06 | train_loss=0.076195 | valid_loss=0.103933
Epoch 07 | train_loss=0.073154 | valid_loss=0.104756
Epoch 08 | train_loss=0.071491 | valid_loss=0.105585
Epoch 09 | train_loss=0.069113 | valid_loss=0.104145
Epoch 10 | train_loss=0.067250 | valid_loss=0.105550

Training history:


,epoch,train_loss,valid_loss
0,1,0.126,0.119
1,2,0.111,0.111
2,3,0.096,0.108
3,4,0.086,0.106
4,5,0.080,0.104
5,6,0.076,0.104
6,7,0.073,0.105
7,8,0.071,0.106
8,9,0.069,0.104
9,10,0.067,0.106


## Candidate-Level Autoencoder Scores

We only need predictions for the held-out validation tracks when comparing fallback strategies.


In [10]:
val_candidate_map = (
    val_df.groupby('UserID')['TrackID']
    .apply(list)
    .to_dict()
)

ae_score_lookup = {}
latent_rows = []

model.eval()
with torch.no_grad():
    for batch_ids, batch_x, _ in iter_dense_batches(ratings_csr, mask_csr, np.arange(ratings_csr.shape[0]), AE_BATCH_SIZE, shuffle=False):
        z = model.encoder(batch_x)
        pred = model.decoder(z).cpu().numpy() * 100.0
        z_np = z.cpu().numpy()

        for local_i, user_row in enumerate(batch_ids):
            uid = ae_user_ids[int(user_row)]
            latent_rows.append({'UserID': uid, **{f'z{k}': float(v) for k, v in enumerate(z_np[local_i])}})
            for tid_str in val_candidate_map.get(uid, []):
                tid_int = int(tid_str)
                col = track_to_idx.get(tid_int)
                if col is not None:
                    ae_score_lookup[(str(uid), str(tid_int))] = float(pred[local_i, col])

latent_df = pd.DataFrame(latent_rows)
print(f'Autoencoder scores stored for validation pairs: {len(ae_score_lookup):,}')
print(f'Latent rows captured                      : {len(latent_df):,}')


Autoencoder scores stored for validation pairs: 130,638
Latent rows captured                      : 34,376


In [11]:
if not latent_df.empty:
    embed_cols = [c for c in latent_df.columns if c.startswith('z')]
    sample_latent = latent_df.sample(min(5000, len(latent_df)), random_state=RANDOM_SEED).copy()
    kmeans = KMeans(n_clusters=8, random_state=RANDOM_SEED, n_init=10)
    sample_latent['cluster'] = kmeans.fit_predict(sample_latent[embed_cols])

    cluster_summary = sample_latent['cluster'].value_counts().sort_index().rename_axis('cluster').reset_index(name='users')
    print('Latent user cluster sizes (sample):')
    display(cluster_summary)

    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    reduced = pca.fit_transform(sample_latent[embed_cols])
    sample_latent['pc1'] = reduced[:, 0]
    sample_latent['pc2'] = reduced[:, 1]
    print('PCA projection sample:')
    display(sample_latent[['UserID', 'cluster', 'pc1', 'pc2']].head(10))
else:
    print('Latent clustering skipped because latent_df is empty.')


Latent user cluster sizes (sample):


,cluster,users
0,0,88
1,1,4084
2,2,332
3,3,307
4,4,63
5,5,81
6,6,22
7,7,23


PCA projection sample:


,UserID,cluster,pc1,pc2
14991,221387,1,-2.230,-0.192
2174,202979,1,-2.440,0.108
6596,209305,1,-2.588,0.512
13583,219381,2,0.998,-3.140
28994,241336,1,-0.665,-2.115
25526,236426,1,-2.407,0.952
1416,201851,1,-2.366,-1.282
15418,221987,1,-2.480,0.456
22206,231702,1,-2.853,0.098
16912,224121,1,0.568,4.245


## Heuristic + Fallback Functions

We compare two album fallback policies:
- `Global`: direct -> sibling tracks -> global Bayesian album score
- `Autoencoder`: direct -> sibling tracks -> AE track score (blended with track popularity) -> global Bayesian


In [12]:
def get_score(item_id, user_ratings, global_item_scores, overall_mean):
    if item_id is None:
        return overall_mean
    if item_id in user_ratings:
        return user_ratings[item_id]
    if item_id in global_item_scores:
        return global_item_scores[item_id]
    return overall_mean


def get_album_score_global(album_id, user_ratings, album_to_tracks, global_item_scores, overall_mean, tracker):
    if album_id is None:
        tracker['no_album_id'] += 1
        return overall_mean, 'no_album_id'

    if album_id in user_ratings:
        tracker['tier1_direct'] += 1
        return user_ratings[album_id], 'tier1_direct'

    sibling_tracks = album_to_tracks.get(album_id, [])
    sibling_ratings = [user_ratings[t] for t in sibling_tracks if t in user_ratings]
    if sibling_ratings:
        tracker['tier2_album_siblings'] += 1
        return float(np.mean(sibling_ratings)), 'tier2_album_siblings'

    tracker['tier3_global'] += 1
    return float(global_item_scores.get(album_id, overall_mean)), 'tier3_global'


def get_album_score_autoencoder(user_id, track_id, album_id, user_ratings, album_to_tracks,
                                global_item_scores, global_track_scores, overall_mean,
                                ae_score_lookup, tracker, blend_weight=0.75):
    if album_id is None:
        tracker['no_album_id'] += 1
        return overall_mean, 'no_album_id'

    if album_id in user_ratings:
        tracker['tier1_direct'] += 1
        return user_ratings[album_id], 'tier1_direct'

    sibling_tracks = album_to_tracks.get(album_id, [])
    sibling_ratings = [user_ratings[t] for t in sibling_tracks if t in user_ratings]
    if sibling_ratings:
        tracker['tier2_album_siblings'] += 1
        return float(np.mean(sibling_ratings)), 'tier2_album_siblings'

    ae_key = (str(user_id), str(track_id))
    if ae_key in ae_score_lookup:
        tracker['tier3_autoencoder'] += 1
        ae_score = float(ae_score_lookup[ae_key])
        track_pop = float(global_track_scores.get(str(int(track_id)), overall_mean))
        blended = blend_weight * ae_score + (1.0 - blend_weight) * track_pop
        return blended, 'tier3_autoencoder'

    tracker['tier4_global'] += 1
    return float(global_item_scores.get(album_id, overall_mean)), 'tier4_global'


def build_feature_row(user_id, track_id, tracker, use_autoencoder=False):
    uid = str(int(user_id))
    tid = str(int(track_id))
    hierarchy = track_to_hierarchy.get(tid, {})
    user_ratings = user_to_ratings.get(uid, {})

    album_id = hierarchy.get('album')
    artist_id = hierarchy.get('artist')
    genre_ids = hierarchy.get('genres', [])

    if use_autoencoder:
        album_score, album_tier = get_album_score_autoencoder(
            user_id=uid,
            track_id=tid,
            album_id=album_id,
            user_ratings=user_ratings,
            album_to_tracks=album_to_tracks,
            global_item_scores=global_item_scores,
            global_track_scores=global_track_scores,
            overall_mean=overall_mean,
            ae_score_lookup=ae_score_lookup,
            tracker=tracker,
            blend_weight=AE_BLEND_WEIGHT,
        )
    else:
        album_score, album_tier = get_album_score_global(
            album_id=album_id,
            user_ratings=user_ratings,
            album_to_tracks=album_to_tracks,
            global_item_scores=global_item_scores,
            overall_mean=overall_mean,
            tracker=tracker,
        )

    artist_score = get_score(artist_id, user_ratings, global_item_scores, overall_mean)
    genre_scores = [get_score(g, user_ratings, global_item_scores, overall_mean) for g in genre_ids]
    ae_track_score = float(ae_score_lookup.get((uid, tid), global_track_scores.get(tid, overall_mean)))
    track_popularity = float(global_track_scores.get(tid, overall_mean))

    if genre_scores:
        genre_count = len(genre_scores)
        genre_max = float(np.max(genre_scores))
        genre_min = float(np.min(genre_scores))
        genre_mean = float(np.mean(genre_scores))
        genre_var = float(np.var(genre_scores))
    else:
        genre_count = 0
        genre_max = overall_mean
        genre_min = overall_mean
        genre_mean = overall_mean
        genre_var = 0.0

    return {
        'UserID': int(user_id),
        'TrackID': tid,
        'album_score': float(album_score),
        'artist_score': float(artist_score),
        'genre_count': int(genre_count),
        'genre_max': float(genre_max),
        'genre_min': float(genre_min),
        'genre_mean': float(genre_mean),
        'genre_var': float(genre_var),
        'ae_track_score': float(ae_track_score),
        'track_popularity': float(track_popularity),
        'album_tier': album_tier,
    }


## Build Validation Feature Matrices

We build one feature matrix per fallback policy.


In [13]:
tracker_global = {
    'tier1_direct': 0,
    'tier2_album_siblings': 0,
    'tier3_global': 0,
    'no_album_id': 0,
}

tracker_auto = {
    'tier1_direct': 0,
    'tier2_album_siblings': 0,
    'tier3_autoencoder': 0,
    'tier4_global': 0,
    'no_album_id': 0,
}

records_global = []
records_auto = []

for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc='Building validation features'):
    base = {
        'true_label': int(row['true_label']),
        'true_rating': float(row['true_rating']),
    }

    fg = build_feature_row(row['UserID'], row['TrackID'], tracker_global, use_autoencoder=False)
    fa = build_feature_row(row['UserID'], row['TrackID'], tracker_auto, use_autoencoder=True)

    fg.update(base)
    fa.update(base)

    records_global.append(fg)
    records_auto.append(fa)

features_global_df = pd.DataFrame(records_global)
features_auto_df = pd.DataFrame(records_auto)

print(f'Global-fallback features : {features_global_df.shape}')
print(f'Autoencoder features     : {features_auto_df.shape}')
print()
print('Sample rows:')
display(features_auto_df.head(6))


Building validation features:   0%|          | 0/130800 [00:00<?, ?it/s]

Global-fallback features : (130800, 14)
Autoencoder features     : (130800, 14)

Sample rows:


,UserID,TrackID,album_score,artist_score,genre_count,genre_max,genre_min,genre_mean,genre_var,ae_track_score,track_popularity,album_tier,true_label,true_rating
0,199810,47420,55.093,48.605,1,43.825,43.825,43.825,0.000,55.995,52.387,tier3_autoencoder,1,90.000
1,199810,283802,50.000,50.000,1,80.000,80.000,80.000,0.000,16.667,31.707,tier1_direct,1,90.000
2,199810,68670,50.000,90.000,3,80.000,39.890,62.817,284.637,32.636,36.229,tier1_direct,1,90.000
3,199810,149988,50.000,90.000,1,80.000,80.000,80.000,0.000,71.841,61.769,tier2_album_siblings,0,50.000
4,199810,29894,53.512,30.000,5,80.000,33.851,49.829,258.829,52.464,56.658,tier3_autoencoder,0,30.000
5,199810,106183,25.435,47.459,5,69.515,28.901,42.105,244.533,15.870,54.131,tier3_autoencoder,0,30.000


## Scoring Rules

We compare a few ranking rules:
- `Weighted Hierarchy`: closest to your original heuristic
- `Adaptive Direct`: direct-only adaptive weighting, with global cold fallback
- `Adaptive + AE Cold`: same as adaptive, but uses the AE score in the no-direct-signal case
- `AE Blend`: explicitly lets the AE score participate in the final score


In [14]:
def score_weighted(row):
    return (
        0.40 * row['album_score'] +
        0.30 * row['artist_score'] +
        0.30 * row['genre_mean']
    )


def score_adaptive_direct(row):
    uid = str(int(row['UserID']))
    tid = str(int(row['TrackID']))
    hierarchy = track_to_hierarchy.get(tid, {})
    user_ratings = user_to_ratings.get(uid, {})

    album_id = hierarchy.get('album')
    artist_id = hierarchy.get('artist')
    genre_ids = hierarchy.get('genres', [])

    weights = {'album': 0.50, 'artist': 0.30, 'genre': 0.20}
    score = 0.0
    total_weight = 0.0

    if album_id and album_id in user_ratings:
        score += weights['album'] * user_ratings[album_id]
        total_weight += weights['album']
    if artist_id and artist_id in user_ratings:
        score += weights['artist'] * user_ratings[artist_id]
        total_weight += weights['artist']

    rated_genres = [user_ratings[g] for g in genre_ids if g in user_ratings]
    if rated_genres:
        score += weights['genre'] * float(np.mean(rated_genres))
        total_weight += weights['genre']

    if total_weight > 0:
        return score / total_weight

    signals = []
    if album_id:
        signals.append(global_item_scores.get(album_id, overall_mean))
    if artist_id:
        signals.append(global_item_scores.get(artist_id, overall_mean))
    for g in genre_ids:
        signals.append(global_item_scores.get(g, overall_mean))
    return float(np.mean(signals)) if signals else overall_mean


def score_adaptive_ae_cold(row):
    uid = str(int(row['UserID']))
    tid = str(int(row['TrackID']))
    hierarchy = track_to_hierarchy.get(tid, {})
    user_ratings = user_to_ratings.get(uid, {})

    album_id = hierarchy.get('album')
    artist_id = hierarchy.get('artist')
    genre_ids = hierarchy.get('genres', [])

    weights = {'album': 0.50, 'artist': 0.30, 'genre': 0.20}
    score = 0.0
    total_weight = 0.0

    if album_id and album_id in user_ratings:
        score += weights['album'] * user_ratings[album_id]
        total_weight += weights['album']
    if artist_id and artist_id in user_ratings:
        score += weights['artist'] * user_ratings[artist_id]
        total_weight += weights['artist']

    rated_genres = [user_ratings[g] for g in genre_ids if g in user_ratings]
    if rated_genres:
        score += weights['genre'] * float(np.mean(rated_genres))
        total_weight += weights['genre']

    if total_weight > 0:
        return score / total_weight

    return 0.80 * row['ae_track_score'] + 0.20 * row['track_popularity']


def score_ae_blend(row):
    return (
        0.30 * row['album_score'] +
        0.25 * row['artist_score'] +
        0.20 * row['genre_mean'] +
        0.25 * row['ae_track_score']
    )


SCORERS = {
    'Weighted Hierarchy': score_weighted,
    'Adaptive Direct': score_adaptive_direct,
    'Adaptive + AE Cold': score_adaptive_ae_cold,
    'AE Blend': score_ae_blend,
}

print('Scorers ready:')
for name in SCORERS:
    print(' ', name)


Scorers ready:
  Weighted Hierarchy
  Adaptive Direct
  Adaptive + AE Cold
  AE Blend


## Evaluation Framework

For each user, rank the 6 held-out tracks and label the top 3 as positive.


In [15]:
def evaluate_pipeline(features_df, score_fn, pipeline_name, scorer_name):
    per_user = []
    labelled_rows = []

    for user_id, group in features_df.groupby('UserID'):
        group = group.copy()
        group['pred_score'] = group.apply(score_fn, axis=1)
        group = group.sort_values('pred_score', ascending=False).reset_index(drop=True)
        group['pred_label'] = [1 if i < 3 else 0 for i in range(len(group))]
        group['correct'] = (group['pred_label'] == group['true_label'])
        hits = int(((group['pred_label'] == 1) & (group['true_label'] == 1)).sum())
        exact = int(group['correct'].all())

        per_user.append({
            'UserID': user_id,
            'correct': int(group['correct'].sum()),
            'hits_at_3': hits,
            'exact_match': exact,
        })
        labelled_rows.append(group)

    results_df = pd.DataFrame(per_user)
    row_acc = float(results_df['correct'].sum() / (len(results_df) * 6))
    mean_hits = float(results_df['hits_at_3'].mean())
    exact_rate = float(results_df['exact_match'].mean())
    labelled_df = pd.concat(labelled_rows, ignore_index=True)

    print() 
    print('=' * 72)
    print(f'{pipeline_name} | {scorer_name}')
    print('=' * 72)
    print(f'Row accuracy        : {row_acc * 100:6.2f}%')
    print(f'Lift vs 50% baseline: {(row_acc - 0.5) * 100:+6.2f} pp')
    print(f'Mean hits@3         : {mean_hits:.3f} / 3')
    print(f'Exact 3-of-3 rate   : {exact_rate * 100:6.2f}%')
    print(f'Users evaluated     : {len(results_df):,}')

    return {
        'row_accuracy': row_acc,
        'mean_hits_at_3': mean_hits,
        'exact_rate': exact_rate,
        'per_user': results_df,
        'labelled_rows': labelled_df,
    }


In [16]:
all_results = []
detailed_results = {}

pipelines = {
    'Global Fallback': features_global_df,
    'Autoencoder Fallback': features_auto_df,
}

for pipeline_name, df in pipelines.items():
    for scorer_name, scorer_fn in SCORERS.items():
        result = evaluate_pipeline(df, scorer_fn, pipeline_name, scorer_name)
        detailed_results[(pipeline_name, scorer_name)] = result
        all_results.append({
            'pipeline': pipeline_name,
            'scorer': scorer_name,
            'row_accuracy': result['row_accuracy'],
            'mean_hits_at_3': result['mean_hits_at_3'],
            'exact_rate': result['exact_rate'],
            'lift_pp': (result['row_accuracy'] - 0.5) * 100,
        })

results_summary_df = pd.DataFrame(all_results).sort_values(
    ['row_accuracy', 'exact_rate', 'mean_hits_at_3'], ascending=False
).reset_index(drop=True)

print()
print('Final ranking summary:')
display(results_summary_df)



Global Fallback | Weighted Hierarchy
Row accuracy        :  79.44%
Lift vs 50% baseline: +29.44 pp
Mean hits@3         : 2.383 / 3
Exact 3-of-3 rate   :  50.57%
Users evaluated     : 21,800

Global Fallback | Adaptive Direct
Row accuracy        :  80.55%
Lift vs 50% baseline: +30.55 pp
Mean hits@3         : 2.416 / 3
Exact 3-of-3 rate   :  54.06%
Users evaluated     : 21,800

Global Fallback | Adaptive + AE Cold
Row accuracy        :  81.17%
Lift vs 50% baseline: +31.17 pp
Mean hits@3         : 2.435 / 3
Exact 3-of-3 rate   :  55.21%
Users evaluated     : 21,800

Global Fallback | AE Blend
Row accuracy        :  81.35%
Lift vs 50% baseline: +31.35 pp
Mean hits@3         : 2.441 / 3
Exact 3-of-3 rate   :  54.34%
Users evaluated     : 21,800

Autoencoder Fallback | Weighted Hierarchy
Row accuracy        :  80.81%
Lift vs 50% baseline: +30.81 pp
Mean hits@3         : 2.424 / 3
Exact 3-of-3 rate   :  53.04%
Users evaluated     : 21,800

Autoencoder Fallback | Adaptive Direct
Row accuracy 

,pipeline,scorer,row_accuracy,mean_hits_at_3,exact_rate,lift_pp
0,Autoencoder Fallback,AE Blend,0.817,2.451,0.549,31.703
1,Global Fallback,AE Blend,0.814,2.441,0.543,31.355
2,Global Fallback,Adaptive + AE Cold,0.812,2.435,0.552,31.168
3,Autoencoder Fallback,Adaptive + AE Cold,0.812,2.435,0.552,31.168
4,Autoencoder Fallback,Weighted Hierarchy,0.808,2.424,0.530,30.812
5,Global Fallback,Adaptive Direct,0.805,2.416,0.541,30.549
6,Autoencoder Fallback,Adaptive Direct,0.805,2.416,0.541,30.549
7,Global Fallback,Weighted Hierarchy,0.794,2.383,0.506,29.437


## Tier Diagnostics

This shows whether the autoencoder is actually rescuing rows that would otherwise fall back to global.


In [17]:
def summarize_tracker(tracker, label):
    total = sum(tracker.values())
    print()
    print(label)
    print('-' * len(label))
    for tier, count in tracker.items():
        pct = (count / total * 100) if total else 0.0
        print(f'  {tier:<22}: {count:>8,}  ({pct:5.1f}%)')
    print(f'  {"TOTAL":<22}: {total:>8,}')

summarize_tracker(tracker_global, 'Global pipeline album tiers')
summarize_tracker(tracker_auto, 'Autoencoder pipeline album tiers')

std_global_pct = tracker_global['tier3_global'] / sum(tracker_global.values()) * 100 if sum(tracker_global.values()) else 0.0
ae_auto_pct = tracker_auto['tier3_autoencoder'] / sum(tracker_auto.values()) * 100 if sum(tracker_auto.values()) else 0.0
ae_global_pct = tracker_auto['tier4_global'] / sum(tracker_auto.values()) * 100 if sum(tracker_auto.values()) else 0.0

print()
print('Key fallback comparison:')
print(f'  Global-only fallback rate     : {std_global_pct:.2f}%')
print(f'  AE rescue rate                : {ae_auto_pct:.2f}%')
print(f'  Remaining true global fallback: {ae_global_pct:.2f}%')



Global pipeline album tiers
---------------------------
  tier1_direct          :   68,002  ( 52.0%)
  tier2_album_siblings  :    7,033  (  5.4%)
  tier3_global          :   47,289  ( 36.2%)
  no_album_id           :    8,476  (  6.5%)
  TOTAL                 :  130,800

Autoencoder pipeline album tiers
--------------------------------
  tier1_direct          :   68,002  ( 52.0%)
  tier2_album_siblings  :    7,033  (  5.4%)
  tier3_autoencoder     :   47,268  ( 36.1%)
  tier4_global          :       21  (  0.0%)
  no_album_id           :    8,476  (  6.5%)
  TOTAL                 :  130,800

Key fallback comparison:
  Global-only fallback rate     : 36.15%
  AE rescue rate                : 36.14%
  Remaining true global fallback: 0.02%


In [18]:
best_pipeline = results_summary_df.iloc[0]['pipeline']
best_scorer = results_summary_df.iloc[0]['scorer']
best_rows = detailed_results[(best_pipeline, best_scorer)]['labelled_rows'].copy()

if 'album_tier' in best_rows.columns:
    tier_perf = (
        best_rows.groupby('album_tier')
        .agg(
            rows=('correct', 'count'),
            correct=('correct', 'sum'),
            avg_true_rating=('true_rating', 'mean'),
        )
        .reset_index()
    )
    tier_perf['row_accuracy'] = tier_perf['correct'] / tier_perf['rows']
    tier_perf = tier_perf.sort_values('row_accuracy', ascending=False)

    print(f'Best configuration: {best_pipeline} | {best_scorer}')
    display(tier_perf)


Best configuration: Autoencoder Fallback | AE Blend


,album_tier,rows,correct,avg_true_rating,row_accuracy
1,tier1_direct,68002,60740,52.840,0.893
3,tier3_autoencoder,47268,35171,43.589,0.744
2,tier2_album_siblings,7033,5171,63.763,0.735
0,no_album_id,8476,5779,53.172,0.682
4,tier4_global,21,7,80.000,0.333


## Optional Next Step for Kaggle

If the autoencoder fallback beats the global fallback on validation, the next move is:
1. rebuild everything on the full `train_data.csv`
2. optionally set `INCLUDE_TEST_CANDIDATES_IN_AE = True`
3. retrain the autoencoder on full training history
4. score `test_data.csv` with the best validation pipeline

This notebook is intentionally focused on the offline experiment first so you can decide if the idea is worth porting back into the main submission pipeline.


In [ ]:
# =========================================
# FINAL KAGGLE RUN: AE AS THE FALLBACK ONLY
# =========================================

FINAL_OUTPUT_CSV = "submission_autoencoder_fallback.csv"

FINAL_TOP_POPULAR_TRACKS = 12000
FINAL_INCLUDE_ALL_TEST_CANDIDATES = True

FINAL_AE_HIDDEN_DIM = 256
FINAL_AE_LATENT_DIM = 64
FINAL_AE_DROPOUT = 0.15
FINAL_AE_EPOCHS = 10
FINAL_AE_BATCH_SIZE = 128
FINAL_AE_LR = 1e-3
FINAL_AE_WEIGHT_DECAY = 1e-5
FINAL_AE_BLEND_WEIGHT = 0.85

print("Running final AE-fallback submission build")
print(f"Output file: {FINAL_OUTPUT_CSV}")


Running final AE-fallback submission build
Output file: submission_autoencoder_fallback.csv


In [20]:
# =========================================
# FULL-DATA LOOKUPS
# =========================================

train_full_df = pd.read_csv("Assets/CSV/train_data.csv")
test_full_df = pd.read_csv("Assets/CSV/test_data.csv")

train_full_df["ItemID_str"] = train_full_df["ItemID"].astype(int).astype(str)
all_track_ids_str = set(str(int(x)) for x in track_df["TrackID"].dropna().astype(int).tolist())

train_full_track_rows = train_full_df[train_full_df["ItemID_str"].isin(all_track_ids_str)].copy()

user_to_ratings_full = defaultdict(dict)
for _, row in tqdm(train_full_df.iterrows(), total=len(train_full_df), desc="Building full user lookup"):
    user_to_ratings_full[str(int(row["UserID"]))][str(int(row["ItemID"]))] = float(row["Rating"])

overall_mean_full = float(train_full_df["Rating"].mean())

track_stats_full = (
    train_full_track_rows.groupby("ItemID")["Rating"]
    .agg(["mean", "count"])
    .reset_index()
)
track_stats_full.columns = ["TrackID", "global_mean", "rating_count"]
track_stats_full["TrackID"] = track_stats_full["TrackID"].astype(int).astype(str)

C_full = float(track_stats_full["rating_count"].median())
m_full = overall_mean_full

track_stats_full["bayesian_avg"] = (
    (track_stats_full["rating_count"] * track_stats_full["global_mean"] + C_full * m_full)
    / (track_stats_full["rating_count"] + C_full)
)

global_track_scores_full = dict(zip(track_stats_full["TrackID"], track_stats_full["bayesian_avg"]))

print(f"overall_mean_full        : {overall_mean_full:.3f}")
print(f"global_track_scores_full : {len(global_track_scores_full):,}")
print(f"C_full                   : {C_full:.1f}")


Building full user lookup:   0%|          | 0/12403575 [00:00<?, ?it/s]

overall_mean_full        : 49.771
global_track_scores_full : 223,780
C_full                   : 9.0


In [21]:
# =========================================
# BUILD AE MATRIX ON FULL TRAINING DATA
# =========================================

test_track_ids_full = set(test_full_df["TrackID"].astype(int).unique().tolist())
popular_track_ids_full = set(
    train_full_track_rows["ItemID"].value_counts().head(FINAL_TOP_POPULAR_TRACKS).index.astype(int).tolist()
)

ae_track_ids_full = set(popular_track_ids_full)
if FINAL_INCLUDE_ALL_TEST_CANDIDATES:
    ae_track_ids_full |= test_track_ids_full

ae_track_ids_full = sorted(ae_track_ids_full)
ae_track_ids_full_str = set(str(t) for t in ae_track_ids_full)

train_full_track_rows_ae = train_full_track_rows[
    train_full_track_rows["ItemID_str"].isin(ae_track_ids_full_str)
].copy()

ae_user_ids_full = sorted(
    set(test_full_df["UserID"].astype(int).unique().tolist()) |
    set(train_full_track_rows_ae["UserID"].astype(int).unique().tolist())
)

user_to_idx_full = {uid: i for i, uid in enumerate(ae_user_ids_full)}
track_to_idx_full = {tid: j for j, tid in enumerate(ae_track_ids_full)}

row_idx_full = train_full_track_rows_ae["UserID"].astype(int).map(user_to_idx_full).to_numpy()
col_idx_full = train_full_track_rows_ae["ItemID"].astype(int).map(track_to_idx_full).to_numpy()
val_full = (train_full_track_rows_ae["Rating"].astype(np.float32) / 100.0).to_numpy()

ratings_csr_full = csr_matrix(
    (val_full, (row_idx_full, col_idx_full)),
    shape=(len(ae_user_ids_full), len(ae_track_ids_full)),
    dtype=np.float32
)

mask_csr_full = ratings_csr_full.copy()
mask_csr_full.data = np.ones_like(mask_csr_full.data, dtype=np.float32)

observed_row_ids_full = np.where(ratings_csr_full.getnnz(axis=1) > 0)[0]
np.random.shuffle(observed_row_ids_full)

val_cut_full = max(1, int(0.10 * len(observed_row_ids_full)))
ae_valid_rows_full = np.sort(observed_row_ids_full[:val_cut_full])
ae_train_rows_full = np.sort(observed_row_ids_full[val_cut_full:])

print(f"AE users full       : {len(ae_user_ids_full):,}")
print(f"AE track vocab full : {len(ae_track_ids_full):,}")
print(f"Observed ratings    : {ratings_csr_full.nnz:,}")
print(f"Density             : {ratings_csr_full.nnz / (ratings_csr_full.shape[0] * ratings_csr_full.shape[1]):.6f}")


AE users full       : 34,378
AE track vocab full : 49,929
Observed ratings    : 3,516,736
Density             : 0.002049


In [22]:
# =========================================
# TRAIN FINAL AUTOENCODER
# =========================================

final_ae_model = TrackAutoencoder(
    input_dim=len(ae_track_ids_full),
    hidden_dim=FINAL_AE_HIDDEN_DIM,
    latent_dim=FINAL_AE_LATENT_DIM,
    dropout=FINAL_AE_DROPOUT,
).to(DEVICE)

final_optimizer = torch.optim.Adam(
    final_ae_model.parameters(),
    lr=FINAL_AE_LR,
    weight_decay=FINAL_AE_WEIGHT_DECAY,
)

def masked_mse_full(pred, target, mask):
    diff = (pred - target) * mask
    denom = torch.clamp(mask.sum(), min=1.0)
    return diff.pow(2).sum() / denom

for epoch in range(1, FINAL_AE_EPOCHS + 1):
    final_ae_model.train()
    train_losses = []

    for _, batch_x, batch_mask in iter_dense_batches(
        ratings_csr_full, mask_csr_full, ae_train_rows_full, FINAL_AE_BATCH_SIZE, shuffle=True
    ):
        final_optimizer.zero_grad()
        pred = final_ae_model(batch_x)
        loss = masked_mse_full(pred, batch_x, batch_mask)
        loss.backward()
        final_optimizer.step()
        train_losses.append(float(loss.item()))

    final_ae_model.eval()
    valid_losses = []
    with torch.no_grad():
        for _, batch_x, batch_mask in iter_dense_batches(
            ratings_csr_full, mask_csr_full, ae_valid_rows_full, FINAL_AE_BATCH_SIZE, shuffle=False
        ):
            pred = final_ae_model(batch_x)
            loss = masked_mse_full(pred, batch_x, batch_mask)
            valid_losses.append(float(loss.item()))

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={np.mean(train_losses):.6f} | "
        f"valid_loss={np.mean(valid_losses):.6f}"
    )


Epoch 01 | train_loss=0.124977 | valid_loss=0.118335
Epoch 02 | train_loss=0.109175 | valid_loss=0.111029
Epoch 03 | train_loss=0.096507 | valid_loss=0.110507
Epoch 04 | train_loss=0.088461 | valid_loss=0.109171
Epoch 05 | train_loss=0.083727 | valid_loss=0.107359
Epoch 06 | train_loss=0.080825 | valid_loss=0.108028
Epoch 07 | train_loss=0.078523 | valid_loss=0.108831
Epoch 08 | train_loss=0.076559 | valid_loss=0.107580
Epoch 09 | train_loss=0.075388 | valid_loss=0.107908
Epoch 10 | train_loss=0.072275 | valid_loss=0.109124


In [23]:
# =========================================
# GENERATE AE SCORES FOR TEST PAIRS
# =========================================

test_candidate_map_full = (
    test_full_df.groupby("UserID")["TrackID"]
    .apply(list)
    .to_dict()
)

ae_score_lookup_test = {}

final_ae_model.eval()
with torch.no_grad():
    all_user_rows_full = np.arange(ratings_csr_full.shape[0])

    for batch_ids, batch_x, _ in iter_dense_batches(
        ratings_csr_full, mask_csr_full, all_user_rows_full, FINAL_AE_BATCH_SIZE, shuffle=False
    ):
        pred = final_ae_model(batch_x).cpu().numpy() * 100.0

        for local_i, user_row in enumerate(batch_ids):
            uid = ae_user_ids_full[int(user_row)]
            for tid in test_candidate_map_full.get(uid, []):
                tid_int = int(tid)
                col = track_to_idx_full.get(tid_int)
                if col is not None:
                    ae_score_lookup_test[(str(uid), str(tid_int))] = float(pred[local_i, col])

print(f"AE scores stored for test pairs: {len(ae_score_lookup_test):,}")
print(f"Total test pairs              : {len(test_full_df):,}")


AE scores stored for test pairs: 120,000
Total test pairs              : 120,000


In [24]:
# =========================================
# AUTOENCODER-FALLBACK SCORING FUNCTION
# =========================================

def get_track_popularity(track_id):
    return float(global_track_scores_full.get(str(int(track_id)), overall_mean_full))


def get_album_score_with_ae_fallback(user_id, track_id, tracker):
    uid = str(int(user_id))
    tid = str(int(track_id))
    user_ratings = user_to_ratings_full.get(uid, {})
    hierarchy = track_to_hierarchy.get(tid, {})

    album_id = hierarchy.get("album")

    if album_id is None:
        tracker["no_album_id"] += 1
        return get_track_popularity(tid)

    if album_id in user_ratings:
        tracker["tier1_direct"] += 1
        return float(user_ratings[album_id])

    sibling_tracks = album_to_tracks.get(album_id, [])
    sibling_ratings = [user_ratings[t] for t in sibling_tracks if t in user_ratings]
    if sibling_ratings:
        tracker["tier2_album_siblings"] += 1
        return float(np.mean(sibling_ratings))

    ae_key = (uid, tid)
    if ae_key in ae_score_lookup_test:
        tracker["tier3_autoencoder"] += 1
        ae_score = float(ae_score_lookup_test[ae_key])
        popularity = get_track_popularity(tid)
        return FINAL_AE_BLEND_WEIGHT * ae_score + (1.0 - FINAL_AE_BLEND_WEIGHT) * popularity

    tracker["tier4_track_popularity"] += 1
    return get_track_popularity(tid)


def score_track_autoencoder_fallback(user_id, track_id, tracker):
    uid = str(int(user_id))
    tid = str(int(track_id))
    user_ratings = user_to_ratings_full.get(uid, {})
    hierarchy = track_to_hierarchy.get(tid, {})

    artist_id = hierarchy.get("artist")
    genre_ids = hierarchy.get("genres", [])

    album_score = get_album_score_with_ae_fallback(uid, tid, tracker)

    if artist_id and artist_id in user_ratings:
        artist_score = float(user_ratings[artist_id])
    else:
        artist_score = float(overall_mean_full)

    rated_genres = [user_ratings[g] for g in genre_ids if g in user_ratings]
    if rated_genres:
        genre_score = float(np.mean(rated_genres))
    else:
        genre_score = float(overall_mean_full)

    # Same structure as your main heuristic, but album branch now falls back to AE
    return (
        0.50 * album_score +
        0.30 * artist_score +
        0.20 * genre_score
    )


In [25]:
# =========================================
# CREATE KAGGLE SUBMISSION CSV
# =========================================

tracker_test = {
    "tier1_direct": 0,
    "tier2_album_siblings": 0,
    "tier3_autoencoder": 0,
    "tier4_track_popularity": 0,
    "no_album_id": 0,
}

predictions = []

for user_id, group in tqdm(
    test_full_df.groupby("UserID"),
    total=test_full_df["UserID"].nunique(),
    desc="Generating submission"
):
    tracks = group["TrackID"].astype(int).tolist()

    scored = []
    for tid in tracks:
        score = score_track_autoencoder_fallback(user_id, tid, tracker_test)
        scored.append((tid, score))

    scored.sort(key=lambda x: x[1], reverse=True)

    for i, (tid, _) in enumerate(scored):
        predictions.append({
            "TrackID": f"{int(user_id)}_{int(tid)}",
            "Predictor": 1 if i < 3 else 0
        })

submission_autoencoder_fallback = pd.DataFrame(predictions)
submission_autoencoder_fallback.to_csv(FINAL_OUTPUT_CSV, index=False)

print(f"Saved: {FINAL_OUTPUT_CSV}")
print(submission_autoencoder_fallback.head(12).to_string(index=False))

# sanity check
submission_autoencoder_fallback["uid"] = submission_autoencoder_fallback["TrackID"].str.split("_").str[0]
bad_users = (submission_autoencoder_fallback.groupby("uid")["Predictor"].sum() != 3).sum()
submission_autoencoder_fallback = submission_autoencoder_fallback.drop(columns=["uid"])

print(f"\nUsers with invalid label counts: {bad_users}")
print(f"Rows in submission             : {len(submission_autoencoder_fallback):,}")
print(f"Total recommend labels         : {submission_autoencoder_fallback['Predictor'].sum():,}")
print(f"Total non-recommend labels     : {(submission_autoencoder_fallback['Predictor'] == 0).sum():,}")


Generating submission:   0%|          | 0/20000 [00:00<?, ?it/s]

Saved: submission_autoencoder_fallback.csv
      TrackID  Predictor
199810_105760          1
 199810_18515          1
 199810_74139          1
199810_208019          0
  199810_9903          0
199810_242681          0
199812_142408          1
199812_130023          1
 199812_29189          1
199812_223706          0
199812_211361          0
199812_276940          0

Users with invalid label counts: 0
Rows in submission             : 120,000
Total recommend labels         : 60,000
Total non-recommend labels     : 60,000


score: 0.849